In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
PROJECT_ROOT = Path.cwd().parents[1]

ML_OUTPUT_DIR = PROJECT_ROOT / "reports" / "ml_outputs"
DCA_OUTPUT_DIR = PROJECT_ROOT / "reports" / "dca_outputs"

PLOT_OUTPUT_DIR = ML_OUTPUT_DIR / "forecast_diagnostics_plots"

In [ ]:
PLOT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
metrics_file = ML_OUTPUT_DIR / "fixed_origin_ml_vs_dca_metrics.csv"

per_well_totals_file = ML_OUTPUT_DIR / "fixed_origin_ml_vs_dca_per_well_totals.csv"

row_level_predictions_file = ML_OUTPUT_DIR / "fixed_origin_ml_vs_baseline_predictions.csv"

In [ ]:
metrics = pd.read_csv(metrics_file)

per_well_totals = pd.read_csv(
    per_well_totals_file,
    dtype={"api8": str},
)

row_level_predictions = pd.read_csv(
    row_level_predictions_file,
    dtype={"api8": str},
)

In [ ]:
metrics.shape, per_well_totals.shape, row_level_predictions.shape

In [ ]:
metrics.head(10)

In [ ]:
forecast_origin_month = 24

forecast_start_month = 25

forecast_end_month = 33

In [ ]:
forecast_window_label = (
    f"Fixed-origin month {forecast_origin_month}; "
    f"forecast months {forecast_start_month}-{forecast_end_month}"
)

In [ ]:
forecast_window_label

In [ ]:
metrics.sort_values("mae_bbl")

In [ ]:
model_label_map = {
    "linear_regression_fixed_origin": "Linear regression",
    "random_forest_fixed_origin": "Random forest",
    "harmonic_dca": "Harmonic DCA",
    "trailing_3mo_fixed_origin": "Trailing 3-mo",
    "naive_fixed_origin": "Naive",
    "hyperbolic_dca": "Hyperbolic DCA",
    "trailing_6mo_fixed_origin": "Trailing 6-mo",
    "exponential_dca": "Exponential DCA",
}

In [ ]:
metrics_plot = metrics.sort_values("mae_bbl").copy()

metrics_plot["model_label"] = metrics_plot["model"].map(model_label_map)

metrics_plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(
    metrics_plot["model_label"],
    metrics_plot["mae_bbl"],
)
axes[0].set_title("MAE by Method")
axes[0].set_xlabel("Method")
axes[0].set_ylabel("MAE, barrels")
axes[0].grid(axis="y", alpha=0.30)
axes[0].tick_params(axis="x", rotation=45)

axes[1].bar(
    metrics_plot["model_label"],
    metrics_plot["wape_percent"],
)
axes[1].set_title("WAPE by Method")
axes[1].set_xlabel("Method")
axes[1].set_ylabel("WAPE, percent")
axes[1].grid(axis="y", alpha=0.30)
axes[1].tick_params(axis="x", rotation=45)

fig.suptitle(f"ML vs DCA Oil Forecast Diagnostics\n{forecast_window_label}")
fig.tight_layout()

plt.show()

In [ ]:
forecast_column_map = {
    "naive_fixed_origin_forecast_test_oil_bbl": "naive_fixed_origin",
    "trailing_3mo_fixed_origin_forecast_test_oil_bbl": "trailing_3mo_fixed_origin",
    "trailing_6mo_fixed_origin_forecast_test_oil_bbl": "trailing_6mo_fixed_origin",
    "linear_regression_forecast_test_oil_bbl": "linear_regression_fixed_origin",
    "random_forest_forecast_test_oil_bbl": "random_forest_fixed_origin",
    "exponential_forecast_test_oil_bbl": "exponential_dca",
    "hyperbolic_forecast_test_oil_bbl": "hyperbolic_dca",
    "harmonic_forecast_test_oil_bbl": "harmonic_dca",
}

In [ ]:
per_well_long = per_well_totals.melt(
    id_vars=[
        "api8",
        "lease_name",
        "well_no",
        "actual_test_oil_bbl",
    ],
    value_vars=list(forecast_column_map.keys()),
    var_name="forecast_column",
    value_name="forecast_test_oil_bbl",
)

In [ ]:
per_well_long["model"] = per_well_long["forecast_column"].map(forecast_column_map)

per_well_long["model_label"] = per_well_long["model"].map(model_label_map)

In [ ]:
per_well_long["error_bbl"] = (
    per_well_long["forecast_test_oil_bbl"]
    - per_well_long["actual_test_oil_bbl"]
)

per_well_long["absolute_error_bbl"] = per_well_long["error_bbl"].abs()

In [ ]:
per_well_long.head()

In [ ]:
per_well_long.tail()

In [ ]:
per_well_long.shape

In [ ]:
selected_scatter_models = [
    "linear_regression_fixed_origin",
    "random_forest_fixed_origin",
    "harmonic_dca",
    "hyperbolic_dca",
]

In [ ]:
scatter_df = per_well_long[
    per_well_long["model"].isin(selected_scatter_models)
].copy()

In [ ]:
scatter_df[
    [
        "api8",
        "lease_name",
        "well_no",
        "actual_test_oil_bbl",
        "model_label",
        "forecast_test_oil_bbl",
        "error_bbl",
    ]
].head(12)

In [ ]:
plot_limit = (
    scatter_df[
        [
            "actual_test_oil_bbl",
            "forecast_test_oil_bbl",
        ]
    ]
    .to_numpy()
    .max()
    * 1.05
)

In [ ]:
plt.figure(figsize=(8, 7))

for model_name, model_df in scatter_df.groupby("model"):
    plt.scatter(
        model_df["actual_test_oil_bbl"],
        model_df["forecast_test_oil_bbl"],
        label=model_label_map[model_name],
        alpha=0.75,
    )

plt.plot(
    [0, plot_limit],
    [0, plot_limit],
    color="black",
    linestyle="--",
    linewidth=1,
)

plt.title(f"Actual vs Forecast Total Oil\n{forecast_window_label}")
plt.xlabel("Actual oil, barrels")
plt.ylabel("Forecast oil, barrels")
plt.xlim(0, plot_limit)
plt.ylim(0, plot_limit)
plt.grid(alpha=0.30)
plt.legend()
plt.tight_layout()

plt.show()

In [ ]:
plt.figure(figsize=(8, 7))

for model_name, model_df in scatter_df.groupby("model"):
    plt.scatter(
        model_df["actual_test_oil_bbl"],
        model_df["forecast_test_oil_bbl"],
        label=model_label_map[model_name],
        alpha=0.75,
    )

plt.plot(
    [0, plot_limit],
    [0, plot_limit],
    color="black",
    linestyle="--",
    linewidth=1,
)

plt.title(f"Actual vs Forecast Total Oil\n{forecast_window_label}")
plt.xlabel("Actual oil, barrels")
plt.ylabel("Forecast oil, barrels")
plt.xlim(0, plot_limit)
plt.ylim(0, plot_limit)
plt.grid(alpha=0.30)
plt.legend()
plt.tight_layout()

plt.savefig(
    PLOT_OUTPUT_DIR / "actual_vs_forecast_scatter_selected_methods.png",
    dpi=150,
)

plt.show()

In [ ]:
selected_error_models = [
    "linear_regression_fixed_origin",
    "random_forest_fixed_origin",
    "harmonic_dca",
]

In [ ]:
error_plot_df = per_well_long[
    per_well_long["model"].isin(selected_error_models)
].copy()

In [ ]:
well_order = (
    error_plot_df[error_plot_df["model"] == "linear_regression_fixed_origin"]
    .sort_values("error_bbl")
    ["api8"]
    .tolist()
)

In [ ]:
error_pivot = (
    error_plot_df.pivot_table(
        index="api8",
        columns="model_label",
        values="error_bbl",
        aggfunc="first",
    )
    .loc[well_order]
)

In [ ]:
error_pivot.head()

In [ ]:
plt.figure(figsize=(14, 6))

error_pivot.plot(kind="bar", ax=plt.gca())

plt.axhline(0, color="black", linewidth=1)
plt.title(f"Per-Well Signed Forecast Error\n{forecast_window_label}")
plt.xlabel("API8")
plt.ylabel("Forecast error, barrels")
plt.grid(axis="y", alpha=0.30)
plt.tight_layout()

plt.show()

In [ ]:
plt.figure(figsize=(14, 6))

error_pivot.plot(kind="bar", ax=plt.gca())

plt.axhline(0, color="black", linewidth=1)
plt.title(f"Per-Well Signed Forecast Error\n{forecast_window_label}")
plt.xlabel("API8")
plt.ylabel("Forecast error, barrels")
plt.grid(axis="y", alpha=0.30)
plt.tight_layout()

plt.savefig(
    PLOT_OUTPUT_DIR / "per_well_signed_forecast_error_selected_methods.png",
    dpi=150,
)

plt.show()

In [ ]:
best_method_by_well = (
    per_well_long.sort_values("absolute_error_bbl")
    .groupby(
        [
            "api8",
            "lease_name",
            "well_no",
        ],
        as_index=False,
    )
    .first()
)

In [ ]:
best_method_by_well = best_method_by_well.sort_values("absolute_error_bbl")

In [ ]:
best_method_by_well[
    [
        "api8",
        "lease_name",
        "well_no",
        "model_label",
        "actual_test_oil_bbl",
        "forecast_test_oil_bbl",
        "error_bbl",
        "absolute_error_bbl",
    ]
].head(10)

In [ ]:
best_method_by_well[
    [
        "api8",
        "lease_name",
        "well_no",
        "model_label",
        "actual_test_oil_bbl",
        "forecast_test_oil_bbl",
        "error_bbl",
        "absolute_error_bbl",
    ]
].tail(10)

In [ ]:
best_wells = best_method_by_well.head(8).copy()

worst_wells = best_method_by_well.tail(8).copy()

In [ ]:
best_worst_plot_df = pd.concat(
    [
        best_wells.assign(group="Best wells"),
        worst_wells.assign(group="Worst wells"),
    ],
    ignore_index=True,
)


In [ ]:
best_worst_plot_df["well_label"] = (
    best_worst_plot_df["api8"]
    + "\n"
    + best_worst_plot_df["model_label"]
)

In [ ]:
best_worst_plot_df[
    [
        "group",
        "api8",
        "lease_name",
        "well_no",
        "model_label",
        "absolute_error_bbl",
    ]
]


In [ ]:
plt.figure(figsize=(13, 6))

colors = np.where(
    best_worst_plot_df["group"] == "Best wells",
    "tab:green",
    "tab:red",
)

plt.bar(
    best_worst_plot_df["well_label"],
    best_worst_plot_df["absolute_error_bbl"],
    color=colors,
)

plt.title(f"Best and Worst Wells by Best Available Absolute Error\n{forecast_window_label}")
plt.xlabel("Well and best method")
plt.ylabel("Absolute error, barrels")
plt.grid(axis="y", alpha=0.30)
plt.tick_params(axis="x", rotation=45)
plt.tight_layout()

plt.show()

In [ ]:
plt.figure(figsize=(13, 6))

colors = np.where(
    best_worst_plot_df["group"] == "Best wells",
    "tab:green",
    "tab:red",
)

plt.bar(
    best_worst_plot_df["well_label"],
    best_worst_plot_df["absolute_error_bbl"],
    color=colors,
)

plt.title(f"Best and Worst Wells by Best Available Absolute Error\n{forecast_window_label}")
plt.xlabel("Well and best method")
plt.ylabel("Absolute error, barrels")
plt.grid(axis="y", alpha=0.30)
plt.tick_params(axis="x", rotation=45)
plt.tight_layout()

plt.savefig(
    PLOT_OUTPUT_DIR / "best_worst_wells_by_absolute_error.png",
    dpi=150,
)

plt.show()

In [ ]:
example_well_api8 = (
    per_well_long[per_well_long["model"] == "linear_regression_fixed_origin"]
    .sort_values("absolute_error_bbl")
    .iloc[len(per_well_totals) // 2]
    ["api8"]
)

example_well_api8


In [ ]:
example_ts = row_level_predictions[
    row_level_predictions["api8"] == example_well_api8
].copy()

example_ts

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    example_ts["target_month_on_production"],
    example_ts["actual_oil_bbl"],
    marker="o",
    label="Actual",
)

plt.plot(
    example_ts["target_month_on_production"],
    example_ts["linear_regression_forecast_oil_bbl"],
    marker="o",
    label="Linear regression",
)

plt.plot(
    example_ts["target_month_on_production"],
    example_ts["random_forest_forecast_oil_bbl"],
    marker="o",
    label="Random forest",
)

plt.plot(
    example_ts["target_month_on_production"],
    example_ts["trailing_3mo_fixed_origin_forecast_oil_bbl"],
    marker="o",
    label="Trailing 3-mo",
)

plt.title(
    f"Example Fixed-Origin ML Forecast Path: API8 {example_well_api8}\n"
    f"Forecast months {forecast_start_month}-{forecast_end_month}"
)
plt.xlabel("Month on production")
plt.ylabel("Oil, barrels")
plt.grid(alpha=0.30)
plt.legend()
plt.tight_layout()

plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    example_ts["target_month_on_production"],
    example_ts["actual_oil_bbl"],
    marker="o",
    label="Actual",
)

plt.plot(
    example_ts["target_month_on_production"],
    example_ts["linear_regression_forecast_oil_bbl"],
    marker="o",
    label="Linear regression",
)

plt.plot(
    example_ts["target_month_on_production"],
    example_ts["random_forest_forecast_oil_bbl"],
    marker="o",
    label="Random forest",
)

plt.plot(
    example_ts["target_month_on_production"],
    example_ts["trailing_3mo_fixed_origin_forecast_oil_bbl"],
    marker="o",
    label="Trailing 3-mo",
)

plt.title(
    f"Example Fixed-Origin ML Forecast Path: API8 {example_well_api8}\n"
    f"Forecast months {forecast_start_month}-{forecast_end_month}"
)
plt.xlabel("Month on production")
plt.ylabel("Oil, barrels")
plt.grid(alpha=0.30)
plt.legend()
plt.tight_layout()

plt.savefig(
    PLOT_OUTPUT_DIR / "example_row_level_ml_forecast_path.png",
    dpi=150,
)

plt.show()

## Notebook Checkpoint

This notebook creates visual diagnostics for fixed-origin oil forecasts.

Completed diagnostics:

- Model comparison bar chart for MAE and WAPE
- Actual vs forecast scatter for selected ML and DCA methods
- Per-well signed forecast error bar chart
- Best and worst wells by absolute error
- Example row-level actual-vs-forecast time series where row-level ML forecast data supports it

Important setup notes:

- Forecast origin: month 24
- Forecast window: months 25-33
- Target: oil only
- DCA comparison-ready data is currently available as per-well forecast totals, not row-level monthly DCA forecasts

In [ ]:
sorted(path.name for path in PLOT_OUTPUT_DIR.glob("*.png"))